In [ ]:
!nvidia-smi

In [ ]:
!pip install -q vllm lm-format-enforcer pandas datasets scikit-learn

In [ ]:
# Need to install this version of protobuf after installing vllm
!pip install "protobuf==5.29.5"

In [ ]:
!pip install -q "datasets<4.0.0"


In [ ]:
from huggingface_hub import login
login()

# RAFT Benchmark — 0-shot Evaluation with Llama-3-8B + vLLM

**RAFT** (Real-world Annotated Few-shot Tasks) is a benchmark of 11 real-world NLP classification tasks.
Each task provides exactly **50 labeled training examples**.
Test labels are withheld — official scoring requires submission to the leaderboard at https://huggingface.co/spaces/ought/raft-leaderboard.

> **Preferred execution:** Use `llama_3_8b_raft_0shot_modal.py` to run this on [Modal](https://modal.com) (no Colab needed):
> ```bash
> pip install modal && modal setup
> modal secret create huggingface-secret HF_TOKEN=<your_token>
> modal run llama_3_8b_raft_0shot_modal.py
> ```
> Download results: `modal volume get raft-results-llama3-8b / ./llama3_8b_raft_results`

### Available tasks
| Task key | Description |
|---|---|
| `ade_corpus_v2` | Adverse drug effects from medical text |
| `banking_77` | Customer service intent (77 classes) |
| `neurips_impact_statement_risks` | NeurIPS paper risk categorization |
| `one_stop_english` | Text readability level |
| `overruling` | Legal — is a rule overruled? |
| `semiconductor_org_types` | Semiconductor organization type |
| `systematic_review_inclusion` | Systematic review paper inclusion |
| `tai_safety_research` | AI safety paper categorization |
| `terms_of_service` | ToS clause policy violation |
| `tweet_eval_hate` | Hate speech detection |
| `twitter_complaints` | Twitter complaint detection |

**Strategy**: Since test labels are withheld, we:
1. Evaluate accuracy on the **50 training examples** (held-out fold)
2. Generate **test predictions** formatted for leaderboard submission


In [ ]:
import os
import re
import json
import time
import numpy as np
from datasets import load_dataset
from tqdm import tqdm

# ── RAFT task metadata ─────────────────────────────────────────────────────────
# Label order matches the official RAFT integer labels (1-indexed).
# label_names[0] -> RAFT label 1, label_names[1] -> RAFT label 2, etc.
# Source: https://huggingface.co/datasets/ought/raft/raw/main/raft.py
RAFT_TASKS = {
    "ade_corpus_v2": {
        "text_fields": ["Sentence"],
        "label_names": ["ADE-related", "not ADE-related"],
        "description": "Classify whether the sentence describes an Adverse Drug Effect (ADE).",
    },
    "banking_77": {
        "text_fields": ["Query"],
        "label_names": [
            "refund not showing up", "activate my card", "age limit",
            "apple pay or google pay", "atm support", "automatic top up",
            "balance not updated after bank transfer",
            "balance not updated after cheque or cash deposit",
            "beneficiary not allowed", "cancel transfer", "card about to expire",
            "card acceptance", "card arrival", "card delivery estimate",
            "card linking", "card not working", "card payment fee charged",
            "card payment not recognised", "card payment wrong exchange rate",
            "card swallowed", "cash withdrawal charge",
            "cash withdrawal not recognised", "change pin", "compromised card",
            "contactless not working", "country support", "declined card payment",
            "declined cash withdrawal", "declined transfer",
            "direct debit payment not recognised", "disposable card limits",
            "edit personal details", "exchange charge", "exchange rate",
            "exchange via app", "extra charge on statement", "failed transfer",
            "fiat currency support", "get disposable virtual card",
            "get physical card", "getting spare card", "getting virtual card",
            "lost or stolen card", "lost or stolen phone", "order physical card",
            "passcode forgotten", "pending card payment", "pending cash withdrawal",
            "pending top up", "pending transfer", "pin blocked", "receiving money",
            "request refund", "reverted card payment",
            "supported cards and currencies", "terminate account",
            "top up by bank transfer charge", "top up by card charge",
            "top up by cash or cheque", "top up failed", "top up limits",
            "top up reverted", "topping up by card", "transaction charged twice",
            "transfer fee charged", "transfer into account",
            "transfer not received by recipient", "transfer timing",
            "unable to verify identity", "verify my identity",
            "verify source of funds", "verify top up", "virtual card not working",
            "visa or mastercard", "why verify identity",
            "wrong amount of cash received", "wrong exchange rate for cash withdrawal",
        ],
        "description": "Classify the customer service query into one of 77 banking intent categories.",
    },
    "neurips_impact_statement_risks": {
        "text_fields": ["Paper title", "Impact statement"],
        "label_names": ["doesn't mention a harmful application", "mentions a harmful application"],
        "description": "Does the NeurIPS impact statement mention a harmful application of the research?",
    },
    "one_stop_english": {
        "text_fields": ["Article"],
        "label_names": ["advanced", "elementary", "intermediate"],
        "description": "Classify the reading level of the article: advanced, elementary, or intermediate.",
    },
    "overruling": {
        "text_fields": ["Sentence"],
        "label_names": ["not overruling", "overruling"],
        "description": "Does this legal sentence overrule a previous legal rule or case?",
    },
    "semiconductor_org_types": {
        "text_fields": ["Paper title", "Organization name"],
        "label_names": ["company", "research institute", "university"],
        "description": "Classify the semiconductor organization type.",
    },
    "systematic_review_inclusion": {
        "text_fields": ["Title", "Abstract"],
        "label_names": ["included", "not included"],
        "description": "Should this paper be included in a systematic review based on its title and abstract?",
    },
    "tai_safety_research": {
        "text_fields": ["Title", "Abstract Note"],
        "label_names": ["TAI safety research", "not TAI safety research"],
        "description": "Is this paper related to transformative AI (TAI) safety research?",
    },
    "terms_of_service": {
        "text_fields": ["Sentence"],
        "label_names": ["not potentially unfair", "potentially unfair"],
        "description": "Does this Terms of Service clause contain a potentially unfair policy?",
    },
    "tweet_eval_hate": {
        "text_fields": ["Tweet"],
        "label_names": ["hate speech", "not hate speech"],
        "description": "Does this tweet contain hate speech?",
    },
    "twitter_complaints": {
        "text_fields": ["Tweet text"],
        "label_names": ["complaint", "no complaint"],
        "description": "Is this tweet a customer complaint?",
    },
}

print(f"RAFT benchmark: {len(RAFT_TASKS)} tasks available")
for task, meta in RAFT_TASKS.items():
    n_labels = len(meta["label_names"])
    print(f"  {task}: {n_labels} label(s), input field(s): {meta['text_fields']}")


In [ ]:
# ── Select task(s) to evaluate ────────────────────────────────────────────────
# Set TASKS_TO_EVAL to a list of task keys, or use ALL_TASKS to run everything.
# Note: banking_77 has 77 classes and may be harder for 0-shot; start with easier tasks.

ALL_TASKS = list(RAFT_TASKS.keys())

TASKS_TO_EVAL = ALL_TASKS # Edit this list, or set TASKS_TO_EVAL = ALL_TASKS to run all 11 tasks

print(f"Tasks to evaluate ({len(TASKS_TO_EVAL)}): {TASKS_TO_EVAL}")

In [ ]:
# ── Load datasets ─────────────────────────────────────────────────────────────
# RAFT splits: 'train' (50 labeled examples), 'test' (unlabeled, labels withheld)
# We evaluate accuracy on 'train' and generate predictions for 'test'.

task_datasets = {}   # task -> {"train": Dataset, "test": Dataset}

for task in TASKS_TO_EVAL:
    print(f"Loading {task}...")
    train_ds = load_dataset("ought/raft", task, split="train")
    test_ds  = load_dataset("ought/raft", task, split="test")
    task_datasets[task] = {"train": train_ds, "test": test_ds}
    print(f"  train: {len(train_ds)} examples | test: {len(test_ds)} examples")
    print(f"  columns: {train_ds.column_names}")
    # RAFT uses integer labels starting at 1 (0 = unlabeled/unknown in some tasks)
    label_vals = sorted(set(train_ds["Label"]))
    print(f"  train label values: {label_vals}")


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
MODEL_NAME      = "meta-llama/Meta-Llama-3-8B-Instruct"
MAX_NEW_TOKENS  = 32          # RAFT labels are short; 32 is sufficient

print({
    "model": MODEL_NAME,
    "max_new_tokens": MAX_NEW_TOKENS,
    "tasks": TASKS_TO_EVAL,
})


In [ ]:
import torch
cuda_ver = torch.version.cuda.replace(".", "")  # e.g. "121"
torch_ver = torch.__version__.split("+")[0]      # e.g. "2.3.0"
print(f"PyTorch {torch_ver}, CUDA {cuda_ver}")

!pip uninstall vllm -y
!pip install vllm --extra-index-url https://download.pytorch.org/whl/cu{cuda_ver}

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
if __name__ == '__main__':
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    print("Loading Llama-3-8B with vLLM...")
    llm = LLM(
        model=MODEL_NAME,
        dtype="bfloat16",
        gpu_memory_utilization=0.95,
        max_model_len=4096,
        enforce_eager=False,
    )

    sampling_params = SamplingParams(
        temperature=0,
        top_k=20,
        max_tokens=MAX_NEW_TOKENS,
    )

    print("Model loaded!")


In [ ]:
# ── Output directory ──────────────────────────────────────────────────────────
IN_COLAB = True
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/RAFT/Llama3_8B_RAFT_Eval_vLLM"
else:
    DRIVE_SAVE_DIR = os.path.abspath("./llama3_8b_raft_eval_vllm_outputs")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving results to: {DRIVE_SAVE_DIR}")


In [ ]:
# ── Prompt builder ─────────────────────────────────────────────────────────────
def build_system_prompt(task: str) -> str:
    meta = RAFT_TASKS[task]
    label_list = ", ".join(f'"{l}"' for l in meta["label_names"])
    return (
        f"You are a text classification assistant. "
        f"{meta['description']} "
        f"Answer with exactly one of the following labels: {label_list}. "
        f"Output only the label, nothing else."
    )


def build_user_content(task: str, example: dict) -> str:
    """Format the example fields as the user message."""
    meta = RAFT_TASKS[task]
    parts = []
    for field in meta["text_fields"]:
        value = example.get(field, "").strip()
        if value:
            parts.append(f"{field}: {value}")
    parts.append("Label:")
    return "\n".join(parts)


def build_prompt(task: str, example: dict) -> str:
    messages = [
        {"role": "system", "content": build_system_prompt(task)},
        {"role": "user",   "content": build_user_content(task, example)},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


# Quick sanity check
sample_task = TASKS_TO_EVAL[0]
sample_ex   = task_datasets[sample_task]["train"][0]
print(f"== Example prompt for task '{sample_task}' ==")
print(build_prompt(sample_task, sample_ex)[:800])


In [ ]:
# ── Label extractor ───────────────────────────────────────────────────────────
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[-1].strip()
    return text


def extract_label(task: str, response: str):
    """
    Map raw model response to a RAFT integer label (1-indexed).
    Returns None if no match found.
    RAFT labels are 1-indexed: label_names[0] -> label 1, label_names[1] -> label 2, etc.
    """
    cleaned = strip_thinking(response).strip().lower()
    label_names = RAFT_TASKS[task]["label_names"]

    # Try exact / substring match, longest-first to avoid partial matches
    sorted_labels = sorted(enumerate(label_names), key=lambda x: -len(x[1]))
    for idx, name in sorted_labels:
        if name.lower() in cleaned:
            return idx + 1  # RAFT is 1-indexed
    return None


# Test extraction
print(extract_label("ade_corpus_v2", "ADE-related"))          # expect 2
print(extract_label("ade_corpus_v2", "not ADE-related"))      # expect 1
print(extract_label("tweet_eval_hate", "not hate speech"))    # expect 1
print(extract_label("tweet_eval_hate", "hate speech"))        # expect 2

In [ ]:
# ── Generate & evaluate — all selected tasks ─────────────────────────────────
from sklearn.metrics import accuracy_score, classification_report

all_task_results = {}   # task -> {"train_results": [...], "test_predictions": [...]}
summary_rows = []

for task in TASKS_TO_EVAL:
    print(f"\n{'='*60}")
    print(f"Task: {task}")
    print(f"{'='*60}")

    train_ds = task_datasets[task]["train"]
    test_ds  = task_datasets[task]["test"]
    meta     = RAFT_TASKS[task]

    # ── Build prompts for train (for accuracy) and test (for submission) ──────
    train_prompts = [build_prompt(task, ex) for ex in train_ds]
    test_prompts  = [build_prompt(task, ex) for ex in test_ds]
    all_prompts   = train_prompts + test_prompts

    print(f"  {len(train_prompts)} train prompts + {len(test_prompts)} test prompts = {len(all_prompts)} total")

    # ── Generate responses ─────────────────────────────────────────────────────
    t0 = time.time()
    outputs = llm.generate(all_prompts, sampling_params)
    gen_time   = time.time() - t0
    total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput   = total_tokens / gen_time if gen_time > 0 else 0.0
    print(f"  Time:       {gen_time/60:.1f} min")
    print(f"  Tokens:     {total_tokens:,}")
    print(f"  Throughput: {throughput:.1f} tokens/sec")

    # ── Score train predictions ────────────────────────────────────────────────
    train_results = []
    for i, (ex, out) in enumerate(zip(train_ds, outputs[:len(train_prompts)])):
        response = out.outputs[0].text
        gt       = int(ex["Label"])          # true label (1-indexed)
        pred     = extract_label(task, response)
        train_results.append({
            "id":           ex.get("ID", i),
            "ground_truth": gt,
            "prediction":   pred,
            "raw_response": response.strip(),
            "is_correct":   (pred == gt) if pred is not None else False,
            "is_unknown":   pred is None,
        })

    correct  = sum(1 for r in train_results if r["is_correct"])
    unknown  = sum(1 for r in train_results if r["is_unknown"])
    accuracy = correct / len(train_results)

    # Accuracy excluding unknowns
    known_pairs = [(r["ground_truth"], r["prediction"])
                   for r in train_results if r["prediction"] is not None]
    acc_known = accuracy_score(
        [p[0] for p in known_pairs], [p[1] for p in known_pairs]
    ) if known_pairs else 0.0

    print(f"\n  Train accuracy (n={len(train_results)}): {accuracy:.4f}  "
          f"(known-only: {acc_known:.4f}, unknown: {unknown})")

    # Classification report
    gt_list   = [r["ground_truth"] for r in train_results if r["prediction"] is not None]
    pred_list = [r["prediction"]   for r in train_results if r["prediction"] is not None]
    if pred_list:
        label_ids   = list(range(1, len(meta["label_names"]) + 1))
        label_names = meta["label_names"]
        # Only report classes that appear in ground truth or predictions
        present = sorted(set(gt_list) | set(pred_list))
        present_names = [label_names[p - 1] for p in present]
        print(classification_report(gt_list, pred_list,
                                     labels=present,
                                     target_names=present_names))

    # ── Collect test predictions (for leaderboard submission) ──────────────────
    test_predictions = []
    for i, (ex, out) in enumerate(zip(test_ds, outputs[len(train_prompts):])):
        response = out.outputs[0].text
        pred     = extract_label(task, response)
        # RAFT submission: if we can't parse, default to label 1 (most common)
        pred_safe = pred if pred is not None else 1
        test_predictions.append({
            "ID":           ex.get("ID", i),
            "Label":        pred_safe,
            "raw_response": response.strip(),
        })

    # ── Save per-task results ──────────────────────────────────────────────────
    task_dir = os.path.join(DRIVE_SAVE_DIR, task)
    os.makedirs(task_dir, exist_ok=True)

    with open(os.path.join(task_dir, "train_results.json"), "w") as f:
        json.dump(train_results, f, indent=2)

    # Submission CSV: ID + Label columns only (RAFT format)
    import csv
    submission_path = os.path.join(task_dir, "test_predictions.csv")
    with open(submission_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["ID", "Label"])
        writer.writeheader()
        for row in test_predictions:
            writer.writerow({"ID": row["ID"], "Label": row["Label"]})

    print(f"  Test predictions saved: {submission_path}")

    # Store for summary
    all_task_results[task] = {
        "train_results":    train_results,
        "test_predictions": test_predictions,
    }
    summary_rows.append({
        "task":              task,
        "n_train":           len(train_results),
        "n_test":            len(test_predictions),
        "accuracy":          round(accuracy, 4),
        "accuracy_known":    round(acc_known, 4),
        "unknown_frac":      round(unknown / len(train_results), 4),
        "n_labels":          len(meta["label_names"]),
        "total_new_tokens":  total_tokens,
        "throughput_tok_per_sec": round(throughput, 1),
        "gen_time_min":      round(gen_time / 60, 2),
    })

In [ ]:
# ── Final summary across tasks ─────────────────────────────────────────────────
import pandas as pd

summary_df = pd.DataFrame(summary_rows)

print("\n" + "="*70)
print("RAFT 0-SHOT EVALUATION SUMMARY")
print(f"Model: {MODEL_NAME}")
print("="*70)
print(summary_df.to_string(index=False))

if len(summary_rows) > 1:
    avg_acc = summary_df["accuracy"].mean()
    avg_acc_known = summary_df["accuracy_known"].mean()
    print(f"\nMean accuracy (all tasks):    {avg_acc:.4f}")
    print(f"Mean accuracy (known only):   {avg_acc_known:.4f}")

# Save summary
SUMMARY_FILE = os.path.join(DRIVE_SAVE_DIR, "raft_summary.json")
final_summary = {
    "model":           MODEL_NAME,
    "tasks":           TASKS_TO_EVAL,
    "per_task":        summary_rows,
    "mean_accuracy":   float(summary_df["accuracy"].mean()) if len(summary_rows) > 1 else summary_rows[0]["accuracy"],
}
with open(SUMMARY_FILE, "w") as f:
    json.dump(final_summary, f, indent=2)
print(f"\nSummary saved to: {SUMMARY_FILE}")

print("\nNote: Test predictions (test_predictions.csv per task) can be submitted")
print("to the RAFT leaderboard: https://huggingface.co/spaces/ought/raft-leaderboard")


In [ ]:
# ── (Optional) Combine test_predictions.csv files into a single submission ───
# The official RAFT submission expects one CSV per task.
# This cell just lists what was generated.

print("Generated submission files:")
for task in TASKS_TO_EVAL:
    task_dir = os.path.join(DRIVE_SAVE_DIR, task)
    csv_path = os.path.join(task_dir, "test_predictions.csv")
    if os.path.exists(csv_path):
        import pandas as pd
        df = pd.read_csv(csv_path)
        print(f"  {task}: {len(df)} rows  →  {csv_path}")